In [8]:
#dxy cleaner

import pandas as pd

# 1. Load the raw DXY dataset from FRED
df = pd.read_csv("DTWEXBGS.csv")

# 2. Clean and format dates
df['observation_date'] = pd.to_datetime(df['observation_date'])
df = df.sort_values('observation_date').reset_index(drop=True)

# 3. Handle market holidays by forward-filling the last available price
df['DTWEXBGS'] = df['DTWEXBGS'].ffill()

# 4. Filter for your exact backtesting time horizon (Jan 2010 to May 2026)
df_filtered = df[(df['observation_date'] >= '2010-01-01') & (df['observation_date'] <= '2026-05-31')].copy()

# 5. Group by Year-Month and extract the row corresponding to the last available business day
df_filtered['YearMonth'] = df_filtered['observation_date'].dt.to_period('M')
monthly_df = df_filtered.loc[df_filtered.groupby('YearMonth')['observation_date'].idxmax()].copy()

# 6. Apply the lag adjustment rule: calculate the exact date of the following Monday
def get_following_monday(dt):
    # dt.weekday() returns: 0 for Monday, 1 for Tuesday, ..., 4 for Friday
    # Calculate exact days ahead to arrive at the subsequent Monday
    days_ahead = 7 - dt.weekday()
    return dt + pd.Timedelta(days=days_ahead)

monthly_df['release_date'] = monthly_df['observation_date'].apply(get_following_monday)

# 7. Restructure and rename columns for a clean engine injection
final_df = monthly_df[['observation_date', 'release_date', 'DTWEXBGS']].copy()
final_df.columns = ['last_business_day', 'release_date', 'DXY_value']
final_df['DXY_3M_Pct_Change'] = final_df['DXY_value'].pct_change(periods=3)

# 8. Save the time-consistent series to a new CSV file
final_df.to_csv("DXY_Processed_Monthly.csv", index=False)

# Display preview of the results
print(final_df.head())

    last_business_day release_date  DXY_value  DXY_3M_Pct_Change
19         2010-01-29   2010-02-01    93.7638                NaN
39         2010-02-26   2010-03-01    93.6602                NaN
62         2010-03-31   2010-04-05    92.8987                NaN
84         2010-04-30   2010-05-03    92.8423          -0.009828
105        2010-05-31   2010-06-07    96.3062           0.028251


In [9]:
#brent crude oil prices and cleaning

import pandas as pd
import numpy as np

# 1. Load the raw Brent Crude dataset
df = pd.read_csv("DCOILBRENTEU.csv")

# 2. Clean data and format dates
# FRED sometimes uses '.' for missing holiday data, so we force them to NaN, then forward-fill
df['observation_date'] = pd.to_datetime(df['observation_date'])
df['DCOILBRENTEU'] = pd.to_numeric(df['DCOILBRENTEU'], errors='coerce')
df = df.sort_values('observation_date').reset_index(drop=True)
df['DCOILBRENTEU'] = df['DCOILBRENTEU'].ffill()

# 3. Filter for the exact backtesting time horizon (Jan 2010 to May 2026)
df_filtered = df[(df['observation_date'] >= '2010-01-01') & (df['observation_date'] <= '2026-05-31')].copy()

# 4. Group by Year-Month and extract the last available business day
df_filtered['YearMonth'] = df_filtered['observation_date'].dt.to_period('M')
monthly_df = df_filtered.loc[df_filtered.groupby('YearMonth')['observation_date'].idxmax()].copy()

# 5. Apply the specific Wednesday batch release logic
def get_wednesday_release(dt):
    # dt.weekday() returns: 0=Mon, 1=Tue, ..., 4=Fri, 6=Sun
    # Rule: Data up to Monday is released on the Wednesday of that same week (+2 days).
    if dt.weekday() == 0: # If it's Monday
        days_ahead = 2
    else:
        # Find days until the *next* Monday, then add 2 days to reach Wednesday
        days_to_next_monday = 7 - dt.weekday()
        days_ahead = days_to_next_monday + 2
        
    return dt + pd.Timedelta(days=days_ahead)

monthly_df['release_date'] = monthly_df['observation_date'].apply(get_wednesday_release)

# 6. Calculate the 3-Month Percentage Change
# We use periods=3 because the data is now monthly
monthly_df['Brent_3M_Pct_Change'] = monthly_df['DCOILBRENTEU'].pct_change(periods=3)

# 7. Restructure and rename columns for clean engine injection
final_df = monthly_df[['observation_date', 'release_date', 'DCOILBRENTEU', 'Brent_3M_Pct_Change']].copy()
final_df.columns = ['last_business_day', 'release_date', 'Brent_Price', 'Brent_3M_Pct_Change']

# 8. Save to CSV
final_df.to_csv("Brent_Processed_Monthly.csv", index=False)

# Display preview of the results
print(final_df.head(10))
print("-" * 50)
print(final_df.tail(5))

    last_business_day release_date  Brent_Price  Brent_3M_Pct_Change
19         2010-01-29   2010-02-03        71.20                  NaN
39         2010-02-26   2010-03-03        76.36                  NaN
62         2010-03-31   2010-04-07        80.37                  NaN
84         2010-04-30   2010-05-05        86.19             0.210534
105        2010-05-31   2010-06-02        73.00            -0.044002
127        2010-06-30   2010-07-07        74.94            -0.067563
149        2010-07-30   2010-08-04        77.50            -0.100824
171        2010-08-31   2010-09-08        75.51             0.034384
193        2010-09-30   2010-10-06        80.77             0.077796
214        2010-10-29   2010-11-03        82.47             0.064129
--------------------------------------------------
     last_business_day release_date  Brent_Price  Brent_3M_Pct_Change
4194        2026-01-30   2026-02-04        72.25             0.104065
4214        2026-02-27   2026-03-04        71.32  

In [10]:
import pandas as pd
import pandas_datareader.data as web
import datetime
import holidays

# ==========================================
# 1. SETUP: MASTER CALENDAR & HOLIDAYS
# ==========================================
master = pd.read_csv("Monthly_Calendar_2010_2026.csv")
master['Last Day of Month'] = pd.to_datetime(master['Last Day of Month'])
master = master.sort_values('Last Day of Month').reset_index(drop=True)

# Indian Maharashtra banking holidays
in_holidays = holidays.India(years=range(2010, 2027), subdiv='MH')

def get_first_business_day_of_next_month(date_obj):
    """
    Returns the first business day of the month AFTER date_obj's month.
    e.g. for any date in January 2024 → first business day of February 2024.
    """
    # Move to the 1st of the next month
    if date_obj.month == 12:
        first_of_next = datetime.date(date_obj.year + 1, 1, 1)
    else:
        first_of_next = datetime.date(date_obj.year, date_obj.month + 1, 1)

    curr = pd.Timestamp(first_of_next)
    while curr.weekday() >= 5 or curr in in_holidays:
        curr += pd.Timedelta(days=1)
    return curr

master['Release_Date'] = master['Last Day of Month'].apply(get_first_business_day_of_next_month)

# ==========================================
# 2. PULL US FED FUNDS (FEDFUNDS)
# ==========================================
# FEDFUNDS is the monthly average effective rate, released by FRED.
# We pull from Oct 2009 so Jan 2010 has a valid backward-snap value.
start = datetime.datetime(2009, 10, 1)
end   = datetime.datetime(2026, 6, 30)

df_fed = web.DataReader('FEDFUNDS', 'fred', start, end).reset_index()
df_fed.columns = ['EOM_Date', 'Fed_Funds_Rate']

# FRED returns the 1st of each month; convert to end-of-month so the
# merge_asof direction='backward' snaps correctly against Last Day of Month.
df_fed['EOM_Date'] = df_fed['EOM_Date'] + pd.offsets.MonthEnd(0)
df_fed = df_fed.sort_values('EOM_Date').reset_index(drop=True)

# ==========================================
# 3. PREPARE RBI REPO RATE
# ==========================================
repo_timeline = {
    '2025-12-05': 5.25, '2025-06-06': 5.50, '2025-04-09': 6.00, '2025-02-07': 6.25,
    '2023-02-08': 6.50, '2022-12-07': 6.25, '2022-09-30': 5.90, '2022-08-05': 5.40,
    '2022-06-08': 4.90, '2022-05-04': 4.40, '2020-05-22': 4.00, '2020-03-27': 4.40,
    '2019-10-04': 5.15, '2019-08-07': 5.40, '2019-06-06': 5.75, '2019-04-04': 6.00,
    '2019-02-07': 6.25, '2018-08-01': 6.50, '2018-06-06': 6.25, '2017-08-02': 6.00,
    '2016-10-04': 6.25, '2016-04-05': 6.50, '2015-09-29': 6.75, '2015-06-02': 7.25,
    '2015-03-04': 7.50, '2015-01-15': 7.75, '2014-01-28': 8.00, '2013-10-29': 7.75,
    '2013-09-20': 7.50, '2013-05-03': 7.25, '2013-03-19': 7.50, '2013-01-29': 7.75,
    '2012-04-17': 8.00, '2011-10-25': 8.50, '2011-09-16': 8.25, '2011-07-26': 8.00,
    '2011-06-16': 7.50, '2011-05-03': 7.25, '2011-03-17': 6.75, '2011-01-25': 6.50,
    '2010-11-02': 6.25, '2010-09-16': 6.00, '2010-07-27': 5.75, '2010-07-02': 5.50,
    '2010-04-20': 5.25, '2010-03-19': 5.00, '2009-04-21': 4.75
}

df_rbi = pd.DataFrame(
    list(repo_timeline.items()),
    columns=['Effective_Date', 'RBI_Repo_Rate']
)
df_rbi['Effective_Date'] = pd.to_datetime(df_rbi['Effective_Date'])
df_rbi = df_rbi.sort_values('Effective_Date').reset_index(drop=True)

# ==========================================
# 4. MERGE — backward snap as-of Last Day of Month
# ==========================================
# Both source DataFrames must be sorted for merge_asof
df = pd.merge_asof(
    master,
    df_fed,
    left_on='Last Day of Month',
    right_on='EOM_Date',
    direction='backward'
)

df = pd.merge_asof(
    df,
    df_rbi,
    left_on='Last Day of Month',
    right_on='Effective_Date',
    direction='backward'
)

# ==========================================
# 5. CALCULATIONS & EXPORT
# ==========================================
df['Fed_RBI_Spread']   = df['RBI_Repo_Rate'] - df['Fed_Funds_Rate']
df['Spread_3M_Change'] = df['Fed_RBI_Spread'].diff(periods=3)

final_df = df[[
    'Last Day of Month',
    'Release_Date',
    'RBI_Repo_Rate',
    'Fed_Funds_Rate',
    'Fed_RBI_Spread',
    'Spread_3M_Change'
]].copy()

final_df.columns = [
    'last_business_day',
    'release_date',
    'RBI_Repo_Rate',
    'Fed_Funds_Rate',
    'Fed_RBI_Spread',
    'Spread_3M_Change'
]

# Round rate columns to 2 decimal places
for col in ['RBI_Repo_Rate', 'Fed_Funds_Rate', 'Fed_RBI_Spread', 'Spread_3M_Change']:
    final_df[col] = final_df[col].round(2)

final_df.to_csv("Fed_RBI_Spread_Final_Standardized.csv", index=False)
print("Standardized dataset created successfully.")
print(final_df.head(10))

Standardized dataset created successfully.
  last_business_day release_date  RBI_Repo_Rate  Fed_Funds_Rate  \
0        2010-01-31   2010-02-01           4.75            0.11   
1        2010-02-28   2010-03-02           4.75            0.13   
2        2010-03-31   2010-04-01           5.00            0.16   
3        2010-04-30   2010-05-03           5.25            0.20   
4        2010-05-31   2010-06-01           5.25            0.20   
5        2010-06-30   2010-07-01           5.25            0.18   
6        2010-07-31   2010-08-02           5.75            0.18   
7        2010-08-31   2010-09-01           5.75            0.19   
8        2010-09-30   2010-10-01           6.00            0.19   
9        2010-10-31   2010-11-01           6.00            0.19   

   Fed_RBI_Spread  Spread_3M_Change  
0            4.64               NaN  
1            4.62               NaN  
2            4.84               NaN  
3            5.05              0.41  
4            5.05            

In [11]:
import pandas as pd

# ==========================================
# CONFIGURATION
# ==========================================
BASE_FILE = 'Monthly_Calendar_2010_2026.csv'

# Updated config structure: (file, matching_date_col, metric_col, output_col_name, strategy)
SOURCE_CONFIG = [
    (
        'Fed_RBI_Spread_Final_Standardized.csv',
        'last_business_day',
        'Spread_3M_Change',
        'Spread_3M_Change',
        'eom'          # Inputted by Month-End / Last Business Day
    ),
    (
        'DXY_Processed_Monthly.csv',
        'release_date',
        'DXY_3M_Pct_Change',
        'DXY_3M_Change',
        'release'      # Inputted dynamically by True Release Date
    ),
    (
        'Brent_Processed_Monthly.csv',
        'release_date',
        'Brent_3M_Pct_Change',
        'Brent_3M_Change',
        'release'      # Inputted dynamically by True Release Date
    ),
    (
        'FPI_Equity_Flows_2010_2026.csv',
        'Month End Date',
        'Net FPI Equity Flow (Rs Cr)',
        'Net_FPI',
        'eom'          # Inputted by Month-End Date column
    ),
]

# ==========================================
# 1. LOAD BASE CALENDAR SPINE
# ==========================================
df_base = pd.read_csv(BASE_FILE)
df_base['Last Day of Month'] = pd.to_datetime(df_base['Last Day of Month'])
df_base = df_base[['Last Day of Month']].sort_values('Last Day of Month').reset_index(drop=True)

# Year-month period helper for the exact 'eom' matches
df_base['_ym'] = df_base['Last Day of Month'].dt.to_period('M')

# ==========================================
# 2. HYBRID MERGE LOOP
# ==========================================
for file, date_col, metric_col, out_name, strategy in SOURCE_CONFIG:
    df_src = pd.read_csv(file)
    df_src = df_src[[date_col, metric_col]].copy()
    df_src[date_col] = pd.to_datetime(df_src[date_col], errors='coerce')

    # Your bulletproof numeric cleaning blocks
    temp_str = df_src[metric_col].astype(str)
    temp_str = temp_str.str.replace(',', '', regex=False)
    temp_str = temp_str.str.replace(r'[−–—]', '-', regex=True)
    temp_str = temp_str.str.replace(r'\s+', '', regex=True)
    temp_str = temp_str.str.replace(r'^\((.*)\)$', r'-\1', regex=True)
    temp_str = temp_str.str.replace(r'[₹$%]', '', regex=True)
    df_src[metric_col] = pd.to_numeric(temp_str, errors='coerce')

    df_src = df_src.dropna(subset=[date_col, metric_col])

    # Execute routing strategy
    if strategy == 'eom':
        # Strategy A: Use Month-End Period matching
        df_src['_ym'] = df_src[date_col].dt.to_period('M')
        df_src = df_src[['_ym', metric_col]].rename(columns={metric_col: out_name})
        df_base = df_base.merge(df_src, on='_ym', how='left')
        
    elif strategy == 'release':
        # Strategy B: Use Point-In-Time Backward Snapping based on Release Date
        df_src = df_src.sort_values(date_col)
        df_src = df_src[[date_col, metric_col]].rename(columns={metric_col: out_name})
        
        df_base = pd.merge_asof(
            df_base,
            df_src,
            left_on='Last Day of Month',
            right_on=date_col,
            direction='backward'
        )
        # Drop the source date column artifact immediately to prevent naming collisions
        if date_col in df_base.columns:
            df_base = df_base.drop(columns=[date_col])

    print(f"  Merged {out_name:<20} ({strategy.upper():<7}) non-null rows: {df_base[out_name].notna().sum()}")

# ==========================================
# 3. FINAL COLUMN SELECTION & EXPORT
# ==========================================
final_df = df_base[[
    'Last Day of Month',
    'Spread_3M_Change',
    'DXY_3M_Change',
    'Brent_3M_Change',
    'Net_FPI'
]].copy()

final_df['Repo_FedFunds_Spread_3M_Change'] = final_df['Spread_3M_Change'].round(4)
final_df['DXY_3M_Change']    = final_df['DXY_3M_Change'].round(4)
final_df['Brent_3M_Change']  = final_df['Brent_3M_Change'].round(4)
final_df['Net_FPI']          = final_df['Net_FPI'].round(2)

final_df.to_csv('External_stress_variables_merged.csv', index=False)

print("\nMerge complete. Output: External_stress_variables_merged.csv")
print(f"Rows: {len(final_df)}  |  Columns: {list(final_df.columns)}")
print()
print(final_df.head(10).to_string(index=False))

  Merged Spread_3M_Change     (EOM    ) non-null rows: 194
  Merged DXY_3M_Change        (RELEASE) non-null rows: 193
  Merged Brent_3M_Change      (RELEASE) non-null rows: 193
  Merged Net_FPI              (EOM    ) non-null rows: 197

Merge complete. Output: External_stress_variables_merged.csv
Rows: 197  |  Columns: ['Last Day of Month', 'Spread_3M_Change', 'DXY_3M_Change', 'Brent_3M_Change', 'Net_FPI', 'Repo_FedFunds_Spread_3M_Change']

Last Day of Month  Spread_3M_Change  DXY_3M_Change  Brent_3M_Change  Net_FPI  Repo_FedFunds_Spread_3M_Change
       2010-01-31               NaN            NaN              NaN   -500.2                             NaN
       2010-02-28               NaN            NaN              NaN   1216.9                             NaN
       2010-03-31               NaN            NaN              NaN  19928.0                             NaN
       2010-04-30              0.41            NaN              NaN   9361.3                            0.41
       201

In [12]:
import pandas as pd
import numpy as np

def calculate_macro_zscores(input_csv, output_csv, window=36):

    # ==========================================
    # 1. LOAD & SORT ASCENDING FOR ROLLING
    # ==========================================
    # Critical: rolling must run on ascending (oldest→newest) data so that
    # each window only contains past values. Sorting descending first
    # (as in some earlier scripts) introduces look-ahead bias — the window
    # for March would pull in April and May instead of January and February.
    df = pd.read_csv(input_csv)
    df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values('Last Day of Month', ascending=True).reset_index(drop=True)

    # ==========================================
    # 2. ROBUST Z-SCORE WITH WINSORIZATION
    # ==========================================
    # Uses median + MAD instead of mean + std — far more resistant to the
    # kind of extreme macro events (GFC, COVID) that would otherwise blow
    # out a standard z-score and make recent normal moves look flat.
    # 1.4826 scaling factor makes MAD consistent with std for normal data.
    # Winsorization clamps anything beyond ±3 rather than letting it run,
    # keeping the score interpretable as a bounded signal.

    def robust_z_winsorized(window_slice):
        current_val = window_slice[-1]  # always the most recent row (ascending sort)

        if np.isnan(current_val):
            return np.nan

        valid = window_slice[~np.isnan(window_slice)]
        if len(valid) == 0:
            return np.nan

        med = np.median(valid)
        mad = np.median(np.abs(valid - med))

        if mad == 0:
            return 0.0 if current_val == med else np.nan

        z = (current_val - med) / (1.4826 * mad)
        return float(np.clip(z, -4.0, 4.0))

    # ==========================================
    # 3. APPLY ROLLING Z-SCORE TO EACH COLUMN
    # ==========================================
    value_cols = [
        'Repo_FedFunds_Spread_3M_Change',
        'DXY_3M_Change',
        'Brent_3M_Change',
        'Net_FPI',
    ]

    z_df = pd.DataFrame()
    z_df['Last Day of Month'] = df['Last Day of Month'].dt.strftime('%Y-%m-%d')

    for col in value_cols:
        z_df[f'{col}_Z'] = (
            df[col]
            .rolling(window=window, min_periods=18)
            .apply(robust_z_winsorized, raw=True)
        )
        non_null = z_df[f'{col}_Z'].notna().sum()
        print(f"  {col:<22} non-null z-scores: {non_null}")

    z_df['Repo_FedFunds_Spread_3M_Change_Z'] = z_df['Repo_FedFunds_Spread_3M_Change_Z'].mul(-1)
    z_df['Net_FPI_Z'] = z_df['Net_FPI_Z'].mul(-1)

    # ==========================================
    # 4. SORT DESCENDING (newest → oldest) & EXPORT
    # ==========================================
    z_df = z_df.sort_values('Last Day of Month', ascending=False).reset_index(drop=True)
    z_df = z_df.round(4)

    z_df.to_csv(output_csv, index=False)
    print(f"\nZ-score file saved to: {output_csv}")
    print(f"Rows: {len(z_df)}  |  Window: {window} months")
    print()
    print(z_df.head(10).to_string(index=False))


calculate_macro_zscores('External_stress_variables_merged.csv', 'Macro_Factors_Z_Scores.csv', window=36)

  Repo_FedFunds_Spread_3M_Change non-null z-scores: 177
  DXY_3M_Change          non-null z-scores: 176
  Brent_3M_Change        non-null z-scores: 176
  Net_FPI                non-null z-scores: 180

Z-score file saved to: Macro_Factors_Z_Scores.csv
Rows: 197  |  Window: 36 months

Last Day of Month  Repo_FedFunds_Spread_3M_Change_Z  DXY_3M_Change_Z  Brent_3M_Change_Z  Net_FPI_Z
       2026-05-31                           -0.0337           0.2764             4.0000     0.9003
       2026-04-30                           -0.0000           0.4945             4.0000     1.7086
       2026-03-31                           -0.2398          -1.1892             1.2784     3.2563
       2026-02-28                            0.0275          -1.2780             1.3012    -0.4535
       2026-01-31                           -0.5506          -0.1509            -0.6682     1.0876
       2025-12-31                           -0.6745           0.2667            -0.1924     0.7138
       2025-11-30      

In [13]:
import pandas as pd
import numpy as np
import openpyxl

def apply_external_stress_formatting(input_csv, output_xlsx):
    # 1. Load the compiled data
    df = pd.read_csv(input_csv)
    
    # 2. Sort with newest dates on top to match standard spreadsheet layouts
    df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values('Last Day of Month', ascending=False).reset_index(drop=True)
    df['Last Day of Month'] = df['Last Day of Month'].dt.strftime('%Y-%m-%d')
    
    # Identify target numeric columns to apply styling (exclude date columns)
    exclude_cols = ['Last Day of Month', 'release_date']
    z_cols = [col for col in df.columns if col not in exclude_cols]

    # 3. Define multi-tier styling logic
    # Positive values = Red (High Stress), Negative values = Green (Low Stress)
    def format_outliers_tiered(series):
        styles = []
        for val in series:
            if pd.isna(val):
                styles.append('')
            
            # --- POSITIVE DEVIATIONS (RED = HIGH EXTERNAL STRESS) ---
            elif val >= 2.0:
                styles.append('background-color: #ff9999; color: #660000; font-weight: bold;')
            elif val >= 1.0:
                styles.append('background-color: #ffe6e6; color: #990000;')
                
            # --- NEGATIVE DEVIATIONS (GREEN = LOW EXTERNAL STRESS) ---
            elif val <= -2.0:
                styles.append('background-color: #99ff99; color: #004d00; font-weight: bold;')
            elif val <= -1.0:
                styles.append('background-color: #e6ffe6; color: #006600;')
                
            # --- NORMAL/BALANCE RANGE ---
            else:
                styles.append('')
        return styles

    # 4. Apply styling and format to 4 decimal places
    styled_df = df.style.apply(format_outliers_tiered, subset=z_cols, axis=0)
    styled_df = styled_df.format({col: "{:.4f}" for col in z_cols})

    # 5. Export to Excel
    styled_df.to_excel(output_xlsx, index=False, engine='openpyxl')
    print(f"Success! Color-coded Excel sheet generated and saved to: {output_xlsx}")

# =========================================================================
# RUN THE PIPELINE: Put your actual file names inside the quotation marks below!
# =========================================================================

INPUT_CSV_FILE = "Macro_Factors_Z_Scores.csv"
OUTPUT_EXCEL_FILE = "External_Stress_Scores.xlsx"

apply_external_stress_formatting(INPUT_CSV_FILE, OUTPUT_EXCEL_FILE)

Success! Color-coded Excel sheet generated and saved to: External_Stress_Scores.xlsx


In [14]:
import pandas as pd
import numpy as np
import os
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================================
# 1. STANDARDIZED 5-POINT CALIBRATION & REGIME MAPPING
# =========================================================================
def map_to_5_point_scale(z_score):
    """
    Standardizes continuous factors into robust deviation bands.
    Strips out extreme statistical noise while identifying true cyclical shifts.
    """
    if pd.isna(z_score): 
        return 0.0
    if z_score >= 1.4: 
        return 1.0     # Strongly Stressed / High Vulnerability
    elif z_score >= 1.0: 
        return 0.75     # Moderately Stressed
    elif z_score >= 0.55: 
        return 0.5     # Moderately Stressed
    elif z_score >= 0.25: 
        return 0.25     # Moderately Stressed
    elif z_score <= -1.4: 
        return -1.0    # Strongly Favorable / High Cushion
    elif z_score <= -1.0: 
        return -0.75    # Strongly Favorable / High Cushion
    elif z_score <= -0.55: 
        return -0.5    # Strongly Favorable / High Cushion
    elif z_score <= -0.25: 
        return -0.25    # Moderately Favorable
    else: 
        return 0.0     # Neutral Anchor

def classify_net_to_7_point_regime(net_score):
    """
    Translates the continuous, dislocated net external stress index into 
    the stable 7-point master scale. Thresholds are optimized to allow 
    severe structural tail risks to pass through multi-variable averages.
    """
    if pd.isna(net_score):   return 0.0
    if net_score >= 0.70:    return 3.0    # Severe External Shock / BoP Crisis Risk
    elif net_score >= 0.40:  return 2.0    # Significant External Vulnerability
    elif net_score >= 0.10:  return 1.0    # Mild External Headwinds
    elif net_score <= -0.70: return -3.0   # Hyper-Abundant Inflows / Currency Surfeit
    elif net_score <= -0.40: return -2.0   # Significant Capital Inflow / Tailwinds
    elif net_score <= -0.10: return -1.0   # Mild Capital Cushion / Accommodative Bias
    else:                    return 0.0    # Neutral Macro Equilibrium

# =========================================================================
# 2. EXTERNAL STRESS INTERACTION & DISLOCATION ENGINE
# =========================================================================
def calculate_composite_external_stress(dxy_z, brent_z, fpi_z, spread_z):
    """
    Evaluates macro factors and applies the structural dislocation engine.
    Returns BOTH the adjusted net score AND a explicit crisis validation flag
    to distinguish a systemic jump-shock from individual variable noise.
    """
    dxy_s    = map_to_5_point_scale(dxy_z)
    brent_s  = map_to_5_point_scale(brent_z)
    fpi_s    = map_to_5_point_scale(fpi_z)     
    spread_s = map_to_5_point_scale(spread_z)  
    
    # Define Sub-Vector Blocks
    fundamental_stress = (dxy_s + brent_s) / 2.0  # Macro Shocks (DXY + Oil)
    buffer_stress      = (fpi_s + spread_s) / 2.0  # Financial Defenses (FPI + Spread)
    
    # Calculate Equal-Weighted Base Net Score
    base_net_score = (dxy_s + brent_s + fpi_s + spread_s) / 4.0
    
    structural_crisis = False
    
    # --- SMART STRUCTURAL DISLOCATION ENGINE WITH CRISIS TRACKING ---
    # Case 2 Override: Systemic Global Shock (e.g., COVID March 2020)
    # Capital flight is severe (+1.0 stress) but Oil is collapsing (-1.0 cushion)
    if (fpi_s == 1.0 and brent_s == -1.0) or (fpi_s == 1.0 and dxy_s == 1.0):
        # FIX: floor raised from 0.65 to 0.75 so a validated crisis actually
        # clears the >=0.70 threshold needed for flash_regime == 3.0.
        # At 0.65 this never reached the top bucket, so every crisis month
        # in the dataset's history (including Mar-2020) fell through to the
        # ordinary median filter below instead of being confirmed immediately.
        adjusted_net_score = max(base_net_score, 0.75)
        structural_crisis = True                       # Authenticate the systemic crisis flag
        
    elif fundamental_stress * buffer_stress < 0:
        divergence_magnitude = abs(fundamental_stress - buffer_stress)
        
        if fundamental_stress > 0 and buffer_stress < 0:
            # Case 1: The Hot Money Trap (High oil/dollar masked by speculative inflows)
            dislocation_modifier = 0.15 * (divergence_magnitude / 2.0)
            adjusted_net_score = base_net_score + dislocation_modifier
        else:
            # Standard Case 2 Fallback for milder divergences
            dislocation_modifier = -0.15 * (divergence_magnitude / 2.0)
            adjusted_net_score = base_net_score + dislocation_modifier
    else:
        adjusted_net_score = base_net_score
        
    return max(-1.0, min(1.0, adjusted_net_score)), structural_crisis

# =========================================================================
# 3. FALSE-ALARM PROOF TIMING ENGINE (DUAL-PATH ASYMMETRIC FILTER)
# =========================================================================
def apply_validated_asymmetric_filter(regimes, crisis_flags):
    """
    Advanced Chronological Timing Engine.
    - Path A (Fast-Attack): Triggers instantly ONLY if a maximum tail shock 
      is explicitly authenticated by a multi-variable structural crisis flag.
    - Path B (Median Fallback): Subjects standard or unvalidated spikes 
      to the strict 2-out-of-3 chronological majority filter to kill false alarms.
    - Hysteresis Layer (Slow-Decay): Forces a controlled step-down floor 
      when exiting an authenticated crisis state to protect against whipsaws.
    """
    confirmed = []
    current_confirmed = 0.0
    
    for i in range(len(regimes)):
        flash = regimes[i]
        is_validated_crisis = crisis_flags[i]
        
        if i < 2:
            current_confirmed = flash if pd.notna(flash) else 0.0
            confirmed.append(current_confirmed)
            continue
            
        # 1. AUTHENTICATED FAST-ATTACK GATE
        # FIX: gate now checks is_validated_crisis directly instead of
        # requiring flash == 3.0/-3.0. The old check made this branch
        # depend on the floor in calculate_composite_external_stress()
        # landing exactly in the top bucket -- if those two numbers ever
        # drifted apart (as they had, 0.65 vs a 0.70 threshold), the
        # override silently stopped firing with no error anywhere.
        # Checking the flag directly means this can't break that way again.
        if is_validated_crisis:
            current_confirmed = flash
            confirmed.append(current_confirmed)
            continue
            
        # 2. ROBUST MEDIAN FALLBACK PATH (Kills normal market fluctuations and single-variable noise)
        window = [regimes[i], regimes[i-1], regimes[i-2]]
        window = [v for v in window if pd.notna(v)]
        
        if len(window) == 0:
            confirmed.append(current_confirmed)
            continue
            
        counts = pd.Series(window).value_counts()
        highest_frequency = counts.iloc[0]
        most_frequent_value = counts.index[0]
        
        if highest_frequency >= 2:
            proposed_state = most_frequent_value
        else:
            proposed_state = float(np.median(window))
            
        # 3. CONTROLLED SLOW-DECAY HYSTERESIS
        # Prevents the portfolio from slamming straight from an emergency back into maximum risk-on
        if current_confirmed == 3.0 and proposed_state < 1.0:
            current_confirmed = 1.0  # Orderly step-down safety floor
        elif current_confirmed == -3.0 and proposed_state > -1.0:
            current_confirmed = -1.0
        else:
            current_confirmed = proposed_state
            
        confirmed.append(current_confirmed)
        
    return confirmed

# =========================================================================
# 4. WORKBOOK PROCESSING & PRESENTATION PIPELINE
# =========================================================================
def process_external_stress_sheet(input_file, output_file):
    if not os.path.exists(input_file):
        print(f"Error: Could not locate file '{input_file}' in your workspace.")
        return

    # Ingest master native Excel file
    df = pd.read_excel(input_file)
    
    # Chronological sort for lookback calculation integrity
    df['_chrono_helper'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values(by='_chrono_helper', ascending=True).reset_index(drop=True)
    
    # Target columns exactly as they appear in your data layer
    dxy_col    = 'DXY_3M_Change_Z'
    brent_col  = 'Brent_3M_Change_Z'
    fpi_col    = 'Net_FPI_Z'
    spread_col = 'Repo_FedFunds_Spread_3M_Change_Z'
    
    # Execute composite scoring layer (unpacks both net score and crisis validation flag)
    scores_and_flags = df.apply(
        lambda r: calculate_composite_external_stress(r[dxy_col], r[brent_col], r[fpi_col], r[spread_col]), 
        axis=1
    )
    df['Net_External_Stress_Score'] = [item[0] for item in scores_and_flags]
    df['Crisis_Flag'] = [item[1] for item in scores_and_flags]
    
    # Run the flash regime mapping
    df['Flash_Regime'] = [classify_net_to_7_point_regime(val) for val in df['Net_External_Stress_Score']]
    
    # Apply the validated asymmetric chronological lookback engine
    df['Confirmed_Regime'] = apply_validated_asymmetric_filter(
        df['Flash_Regime'].tolist(), 
        df['Crisis_Flag'].tolist()
    )
    
    # Reverse sort to production standard presentation format (latest dates on top)
    df_desc = df.sort_values(by='_chrono_helper', ascending=False).reset_index(drop=True)
    df_desc = df_desc.drop(columns=['_chrono_helper', 'Flash_Regime', 'Crisis_Flag'])  # Clean up processing artifacts
    
    # ---------------------------------------------------------------------
    # PRESENTATION LAYER RE-LABELING
    # ---------------------------------------------------------------------
    presentation_column_mapping = {
        'Last Day of Month': 'Last Day of Month',
        'DXY_3M_Change_Z': 'DXY Momentum Stress (Z)',
        'Brent_3M_Change_Z': 'Oil Shock Momentum Stress (Z)',
        'Net_FPI_Z': 'Capital Outflow Stress (Z)',
        'Repo_FedFunds_Spread_3M_Change_Z': 'Spread Compression Stress (Z)',
        'Net_External_Stress_Score': 'Net External Stress Score',
        'Confirmed_Regime': 'Confirmed External Stress Regime'
    }
    df_desc = df_desc.rename(columns=presentation_column_mapping)
    
    # Output presentation workbook
    df_desc.to_excel(output_file, index=False, sheet_name="Confirmed External Stress")
    
    # Institutional openpyxl Formatting Pipeline
    wb = openpyxl.load_workbook(output_file)
    ws = wb.active
    ws.views.sheetView[0].showGridLines = True
    
    header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid") # Deep Steel Blue
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
    )
    
    for col_idx in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border
        
        col_letter = openpyxl.utils.get_column_letter(col_idx)
        ws.column_dimensions[col_letter].width = 26
        
        for row_idx in range(2, ws.max_row + 1):
            data_cell = ws.cell(row=row_idx, column=col_idx)
            data_cell.border = thin_border
            
            if isinstance(data_cell.value, (int, float)):
                data_cell.alignment = Alignment(horizontal="right")
                data_cell.number_format = '0.00'
            elif isinstance(data_cell.value, pd.Timestamp):
                data_cell.alignment = Alignment(horizontal="center")
                data_cell.number_format = 'yyyy-mm-dd'

    wb.save(output_file)
    print(f"Success! Output compiled to presentation layout with validated asymmetric filter: '{output_file}'")

# =========================================================================
# 5. EXECUTION ENTRYPOINT
# =========================================================================
if __name__ == "__main__":
    SOURCE_DATA = "External_Stress_Scores.xlsx"
    OUTPUT_FILE = "Output_Confirmed_Smooth_External_Stress.xlsx"
    
    process_external_stress_sheet(input_file=SOURCE_DATA, output_file=OUTPUT_FILE)

Success! Output compiled to presentation layout with validated asymmetric filter: 'Output_Confirmed_Smooth_External_Stress.xlsx'
